# LangGraph — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langgraph.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/langgraph.ipynb)

Companion to: [LangGraph](https://anacodicai.org/learn/ai-agents/6-2/langgraph)


## What you'll build

This is the **real `langgraph` library** — not a simulation. We build a small
grading-style state graph with:

1. A node that computes a confidence score.
2. A **conditional edge** that routes to a human-review pause when confidence
   is low, and straight to persistence when it's high.
3. A **checkpointed interrupt/resume**: the graph genuinely pauses execution,
   and a *separate* `.invoke()` call resumes it exactly where it left off —
   the same pattern terrier-ta's real grading pipeline uses.

**No API key needed** — the "LLM" is a plain Python function standing in for
one, so the graph mechanics are 100% real while the "AI" step is a mock.
`pip install langgraph` if running locally; Colab needs it too (see Setup).

### Setup

In [1]:
# %pip install -q langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

print("real langgraph imported")

real langgraph imported


## Step 1 — the state

A `TypedDict` is the graph's shared memory — every node reads from it and
returns a partial update that gets merged in.

In [2]:
class State(TypedDict, total=False):
    text: str
    confidence: float
    grade: str
    override: str

print("State schema defined")

State schema defined


## Step 2 — the nodes

`grade_node` calls our mock "LLM" (swap in a real API call here and nothing
else about the graph changes). `human_review_node` is where the real
LangGraph mechanism lives: `interrupt(...)` genuinely suspends execution.
`persist_node` uses the override if one exists, otherwise the original grade.

In [3]:
def mock_llm_grade(text: str) -> dict:
    # stand-in for a real LLM call -- swap this one function and the
    # rest of the graph is unchanged
    if "great job" in text.lower():
        return {"grade": "A", "confidence": 0.95}
    return {"grade": "C", "confidence": 0.4}

def grade_node(state: State) -> State:
    result = mock_llm_grade(state["text"])
    return {"grade": result["grade"], "confidence": result["confidence"]}

def route_after_grade(state: State) -> str:
    return "human_review" if state["confidence"] < 0.75 else "persist"

def human_review_node(state: State) -> State:
    decision = interrupt({"message": "low confidence, please review", "grade": state["grade"]})
    return {"override": decision}

def persist_node(state: State) -> State:
    final = state.get("override") or state["grade"]
    return {"grade": final}

print("nodes defined")

nodes defined


## Step 3 — wire the graph

**Try it:** change the confidence threshold in `route_after_grade` from `0.75`
to `0.3` and re-run everything below — the "great job" case should stop
pausing entirely.

In [4]:
g = StateGraph(State)
g.add_node("grade", grade_node)
g.add_node("human_review", human_review_node)
g.add_node("persist", persist_node)

g.add_edge(START, "grade")
g.add_conditional_edges("grade", route_after_grade,
                         {"human_review": "human_review", "persist": "persist"})
g.add_edge("human_review", "persist")
g.add_edge("persist", END)

checkpointer = InMemorySaver()
graph = g.compile(checkpointer=checkpointer)
print("graph compiled with a real checkpointer")

graph compiled with a real checkpointer


## Step 4 — a confident case: no pause at all

In [5]:
confident_config = {"configurable": {"thread_id": "demo-confident"}}
result_confident = graph.invoke(
    {"text": "great job on this submission"}, config=confident_config
)
print("confident case result:", result_confident)
assert "__interrupt__" not in result_confident
print("assert passed: high-confidence case never paused")

confident case result: {'text': 'great job on this submission', 'confidence': 0.95, 'grade': 'A'}
assert passed: high-confidence case never paused


## Step 5 — a low-confidence case: it genuinely pauses

Note the `__interrupt__` key in the result — that's LangGraph telling us
execution stopped mid-graph and is waiting.

In [6]:
low_conf_config = {"configurable": {"thread_id": "demo-low-confidence"}}
result_paused = graph.invoke(
    {"text": "the student's answer was okay"}, config=low_conf_config
)
print("paused result:", result_paused)
assert "__interrupt__" in result_paused
print("assert passed: low-confidence case genuinely paused for human review")

paused result: {'text': "the student's answer was okay", 'confidence': 0.4, 'grade': 'C', '__interrupt__': [Interrupt(value={'message': 'low confidence, please review', 'grade': 'C'}, id='f5f46ccf87fe16690fc6f1af81408fc1')]}
assert passed: low-confidence case genuinely paused for human review


## Step 6 — resume it, in a SEPARATE call, with a human decision

This is the part that isn't a simulation: `graph.invoke(Command(resume=...))`
picks the run back up from its checkpoint — the same `thread_id`, a brand new
Python call, and the graph continues exactly where `human_review_node` left
off, with `interrupt(...)`'s return value now set to whatever we resume with.

In [7]:
resumed = graph.invoke(
    Command(resume="B+ (instructor override)"), config=low_conf_config
)
print("after resume:", resumed)

assert resumed["grade"] == "B+ (instructor override)"
assert resumed["override"] == "B+ (instructor override)"
print("assert passed: the resumed run applied the human's override correctly")

after resume: {'text': "the student's answer was okay", 'confidence': 0.4, 'grade': 'B+ (instructor override)', 'override': 'B+ (instructor override)'}
assert passed: the resumed run applied the human's override correctly


## Try it yourself

- Add a THIRD node between `grade` and `persist` for, say, `deduction_memory`
  lookup — this is exactly the shape of terrier-ta's real grading graph
  (`ingest_ocr → load_rubric → query_deduction_mem → grade_attempt →
  self_consistency → gate_human_review → persist`), just with more steps.
- Swap `mock_llm_grade` for a real API call (OpenAI, Anthropic, Groq) — the
  graph wiring, the conditional routing, and the interrupt/resume mechanics
  don't change at all. That's the actual point of LangGraph: the orchestration
  layer is independent of which model is doing the "thinking" inside a node.
- Try calling `graph.invoke(Command(resume=...))` on a `thread_id` that was
  never paused — see what error you get, and think about why a real system
  needs to check `__interrupt__` before ever calling resume.